In [27]:
# Loading the main dataset

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

df = con.sql("""

SELECT *

FROM read_csv_auto(
    'C:/Users/Lenovo/supply-chain-capstone-project/data/raw/DataCoSupplyChainDataset.csv',
    ignore_errors=true,
    all_varchar=true
)

""").df()

print(df.shape)

df.head()

(114187, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.25,314.6400146,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,DEBIT,3,4,22.86000061,304.8099976,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
2,PAYMENT,2,4,134.2100067,298.25,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class
3,TRANSFER,6,4,18.57999992,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,NaN,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/19/2018 11:03,Standard Class
4,DEBIT,2,1,95.18000031,288.4200134,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:42,First Class


In [28]:
con.sql("""
SELECT COUNT(*) AS total_rows
FROM read_csv_auto(
    'C:/Users/Lenovo/supply-chain-capstone-project/data/raw/DataCoSupplyChainDataset.csv',
    ignore_errors=true
)
""").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│     180519 │
└────────────┘



In [29]:
# Loading the logs dataset

logs_df = con.sql("""

SELECT *

FROM read_csv_auto(
    'C:/Users/Lenovo/supply-chain-capstone-project/data/raw/tokenized_access_logs.csv',
    ignore_errors=true,
    all_varchar=true
)

""").df()

print(logs_df.shape)

logs_df.head()

(469977, 8)


,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...
3,Under Armour Men's Compression EV SL Slide,electronics,9/1/2017 6:00,Sep,6,footwear,206.56.112.1,/department/footwear/category/electronics/prod...
4,Pelican Sunstream 100 Kayak,water sports,9/1/2017 6:01,Sep,6,fan shop,136.108.56.242,/department/fan%20shop/category/water%20sports...


In [30]:
clean_df = con.sql("""
SELECT 

    CAST("Order Id" AS VARCHAR) AS order_id,
    CAST("Customer Id" AS VARCHAR) AS customer_id,

    LOWER(TRIM("Product Name")) AS product_name,
    LOWER(TRIM("Category Name")) AS category,
    "Department Name" AS department,

    "Product Price",
    "Product Image" AS product_image,

    "Order Region",
    "Market",
    "Order Status",

    "Shipping Mode",

    CAST("Days for shipping (real)" AS DOUBLE) AS actual_days,
    CAST("Days for shipment (scheduled)" AS DOUBLE) AS scheduled_days,

    COALESCE(CAST("Sales" AS DOUBLE), 0) AS sales,
    CAST("Order Item Quantity" AS INTEGER) AS quantity,
    CAST("Order Profit Per Order" AS DOUBLE) AS profit,

    "Latitude",
    "Longitude",

    (CAST("Days for shipping (real)" AS DOUBLE) 
     - CAST("Days for shipment (scheduled)" AS DOUBLE)) AS delay

FROM read_csv_auto(
    'C:/Users/Lenovo/supply-chain-capstone-project/data/raw/DataCoSupplyChainDataset.csv',
    ignore_errors=true
)
""").df()

In [31]:
#creating demand features from logs dataset
demand_df = con.sql("""
SELECT 
    LOWER(TRIM(Product)) AS product_name,
    COUNT(*) AS view_count,
    COUNT(DISTINCT ip) AS unique_users,
    MODE() WITHIN GROUP (ORDER BY Month) AS peak_month,
    MODE() WITHIN GROUP (ORDER BY Hour) AS peak_hour
FROM read_csv_auto(
    'C:/Users/Lenovo/supply-chain-capstone-project/data/raw/tokenized_access_logs.csv',
    ignore_errors=true
)
GROUP BY product_name
""").df()

In [32]:
merged_df = clean_df.merge(
    demand_df,
    on="product_name",
    how="left"
)

merged_df["view_count"] = merged_df["view_count"].fillna(0)
merged_df["unique_users"] = merged_df["unique_users"].fillna(0)

merged_df.head(50)

,order_id,customer_id,product_name,category,department,Product Price,product_image,Order Region,Market,Order Status,...,sales,quantity,profit,Latitude,Longitude,delay,view_count,unique_users,peak_month,peak_hour
0,77202,20755,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Southeast Asia,Pacific Asia,COMPLETE,...,327.750000,1,91.250000,18.251453,-66.037056,-1.0,0.0,0.0,NaN,NaN
1,75939,19492,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,PENDING,...,327.750000,1,-249.089996,18.279451,-66.037064,1.0,0.0,0.0,NaN,NaN
2,75938,19491,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,CLOSED,...,327.750000,1,-247.779999,37.292233,-121.881279,0.0,0.0,0.0,NaN,NaN
3,75937,19490,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,COMPLETE,...,327.750000,1,22.860001,34.125946,-118.291016,-1.0,0.0,0.0,NaN,NaN
4,75936,19489,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,PENDING_PAYMENT,...,327.750000,1,134.210007,18.253769,-66.037048,-2.0,0.0,0.0,NaN,NaN
5,75935,19488,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,CANCELED,...,327.750000,1,18.580000,43.013969,-78.879066,2.0,0.0,0.0,NaN,NaN
6,75934,19487,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Eastern Asia,Pacific Asia,COMPLETE,...,327.750000,1,95.180000,18.242538,-66.037056,1.0,0.0,0.0,NaN,NaN
7,75933,19486,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Eastern Asia,Pacific Asia,PROCESSING,...,327.750000,1,68.430000,25.928869,-80.162872,1.0,0.0,0.0,NaN,NaN
8,75932,19485,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Eastern Asia,Pacific Asia,CLOSED,...,327.750000,1,133.720001,18.233223,-66.037056,1.0,0.0,0.0,NaN,NaN
9,75931,19484,smart watch,sporting goods,Fitness,327.750000,http://images.acmesports.sports/Smart+watch,Eastern Asia,Pacific Asia,CLOSED,...,327.750000,1,132.149994,37.773991,-121.966629,1.0,0.0,0.0,NaN,NaN


In [33]:
merged_df.to_parquet(
    "C:/Users/Lenovo/supply-chain-capstone-project/data/clean/supply_chain.parquet",
    index=False
)

In [34]:
merged_df.to_csv(
    "C:/Users/Lenovo/supply-chain-capstone-project/data/clean/merged_df.csv",
    index=False
)

FINAL GRAPH SCHEMA

NODES:-    
    
Customer
    
Order
    
Product
    
Category
    
Department
    
Region
    
Market
    
Payment

Relationships

Customer ──PLACED──> Order

Order ──CONTAINS──> Product

Product ──BELONGS_TO──> Category

Product ──PART_OF──> Department

Order ──SHIPPED_TO──> Region

Order ──IN_MARKET──> Market

Order ──PAID_VIA──> Payment

In [35]:
merged_df[[
    "customer_id"
]].drop_duplicates().to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\customers.csv", index=False)

In [36]:
merged_df[[
    "order_id", "Shipping Mode", "Order Region", "Order Status", "delay"
]].drop_duplicates().to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\orders.csv", index=False)

In [37]:
merged_df[[
    "product_name", "category", "department",
    "Product Price", "product_image",
    "view_count", "unique_users",
    "peak_month", "peak_hour"
]].drop_duplicates().to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\products.csv", index=False)

In [38]:
merged_df[["category"]].drop_duplicates() \
    .to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\categories.csv", index=False)

In [39]:
merged_df[["department"]].drop_duplicates() \
    .to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\departments.csv", index=False)

In [40]:
merged_df[["Order Region"]].drop_duplicates() \
    .to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\regions.csv", index=False)

In [41]:
merged_df[["Market"]].drop_duplicates() \
    .to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\markets.csv", index=False)

In [42]:
payment_df = con.sql("""
SELECT DISTINCT Type AS payment_type
FROM read_csv_auto(
    'C:/Users/Lenovo/supply-chain-capstone-project/data/raw/DataCoSupplyChainDataset.csv',
    ignore_errors=true
)
""").df()

payment_df.to_csv(r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\payments.csv", index=False)

In [43]:
#Create relationship CSV's

BASE_PATH = "C:/Users/Lenovo/supply-chain-capstone-project/data/clean"

merged_df[["customer_id", "order_id"]] \
    .to_csv(f"{BASE_PATH}/customer_order.csv", index=False)

merged_df[["order_id", "product_name"]] \
    .to_csv(f"{BASE_PATH}/order_product.csv", index=False)

merged_df[["product_name", "category"]] \
    .drop_duplicates() \
    .to_csv(f"{BASE_PATH}/product_category.csv", index=False)

merged_df[["product_name", "department"]] \
    .drop_duplicates() \
    .to_csv(f"{BASE_PATH}/product_department.csv", index=False)

merged_df[["order_id", "Order Region"]] \
    .to_csv(f"{BASE_PATH}/order_region.csv", index=False)

merged_df[["order_id", "Market"]] \
    .to_csv(f"{BASE_PATH}/order_market.csv", index=False)

In [44]:
# Convert important numeric columns

import pandas as pd
numeric_cols = [
    "Sales",
    "Order Profit Per Order",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Order Item Quantity",
    "Order Item Discount"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [45]:
# Handle missing numeric values

df["Sales"] = df["Sales"].fillna(0)

df["Order Profit Per Order"] = (
    df["Order Profit Per Order"]
    .fillna(0)
)

df["Order Item Quantity"] = (
    df["Order Item Quantity"]
    .fillna(0)
)

df["Order Item Discount"] = (
    df["Order Item Discount"]
    .fillna(0)
)

In [46]:
# Create delay column

df["delay"] = (
    df["Days for shipping (real)"]
    -
    df["Days for shipment (scheduled)"]
)

df[[
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "delay"
]].head()

,Days for shipping (real),Days for shipment (scheduled),delay
0,3,4,-1
1,3,4,-1
2,2,4,-2
3,6,4,2
4,2,1,1


In [47]:
# Create late delivery flag

df["late_flag"] = (
    df["delay"] > 0
).astype(int)

df[[
    "delay",
    "late_flag"
]].head()

,delay,late_flag
0,-1,0
1,-1,0
2,-2,0
3,2,1
4,1,1


In [48]:
df.shape

(114187, 55)

In [49]:
# Remove duplicate rows

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (114187, 55)


In [51]:
# Check remaining missing values in final merged dataset

nulls = merged_df.isnull().sum()

nulls = nulls[nulls > 0]

print(nulls.sort_values(ascending=False))

peak_month    12532
peak_hour     12532
dtype: int64


In [52]:
# Check all columns in merged_df

print(merged_df.columns.tolist())

['order_id', 'customer_id', 'product_name', 'category', 'department', 'Product Price', 'product_image', 'Order Region', 'Market', 'Order Status', 'Shipping Mode', 'actual_days', 'scheduled_days', 'sales', 'quantity', 'profit', 'Latitude', 'Longitude', 'delay', 'view_count', 'unique_users', 'peak_month', 'peak_hour']


In [53]:
# Check final dataset shape

print(merged_df.shape)

(180519, 23)


In [54]:
merged_df["peak_month"] = merged_df["peak_month"].fillna("unknown")

merged_df["peak_hour"] = merged_df["peak_hour"].fillna(-1)

In [55]:
# Clean important categorical columns

categorical_cols = [
    "Order Status",
    "Shipping Mode",
    "Category Name",
    "Department Name",
    "Order Region",
    "Product Name"
]

for col in categorical_cols:
    df[col] = (
        df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

In [56]:
# Remove rows with missing business keys

df = df.dropna(subset=[
    "Order Id",
    "Customer Id",
    "Product Name"
])

print("Shape after removing invalid rows:", df.shape)

Shape after removing invalid rows: (114187, 55)


In [57]:
# Final null check

print(df.isnull().sum())

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        7
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      2
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


In [59]:
# Export final cleaned dataset

df.to_csv(
    r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\merged_df.csv",
    index=False
)

df.to_parquet(
    r"C:\Users\Lenovo\supply-chain-capstone-project\data\clean\merged_df.parquet",
    index=False
)

print("Final cleaned dataset exported successfully.")

Final cleaned dataset exported successfully.
